# 🗣️ Linguistic Agent Training (DistilBERT on Existing Transcripts)

Fine-tunes a **DistilBERT** classification model on an **existing dataset of 5,000 transcripts** (`40_PER_22_Data/transcripts.csv`) in Google Drive to bypass any transcription/ASR failures.

### Features:
1. **Bypasses ASR Pipeline**: Trains directly on pre-extracted transcripts to instantly fix inference issues.
2. **Robust Column & Label Auto-Mapping**: Automatically detects text and label columns, cleaning and mapping strings like `bonafide`/`spoof` to `0`/`1`.
3. **Stratified 80/10/10 Splitting**: Guarantees healthy label distribution in train, dev, and eval sets.

In [ ]:
# Install dependencies
!pip install -q transformers datasets evaluate accelerate torch torchaudio pandas scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

DRIVE_DIR = Path('/content/drive/MyDrive/142_Feature_Extracted')
OLD_TRANSCRIPTS_CSV = Path('/content/drive/MyDrive/40_PER_22_Data/transcripts.csv')
MODEL_SAVE_DIR = DRIVE_DIR / 'Linguistic' / 'Model' / 'saved_model'
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Load, Map, and Clean the Existing Transcripts

In [ ]:
if not OLD_TRANSCRIPTS_CSV.exists():
    raise FileNotFoundError(f"❌ ERROR: Could not find transcripts file at {OLD_TRANSCRIPTS_CSV}!")

df = pd.read_csv(OLD_TRANSCRIPTS_CSV)
print(f"Raw dataset loaded. Total rows: {len(df):,}")
print("Columns present:", list(df.columns))

# 1. Robustly map the transcript text column
text_col = None
for col in ['text', 'transcript', 'transcription', 'transcripts', 'sentence']:
    if col in df.columns:
        text_col = col
        break
if text_col is None:
    # Fallback to the first string/object column
    text_col = df.select_dtypes(include=['object']).columns[0]
print(f"Using text column: '{text_col}'")
df = df.rename(columns={text_col: 'text'})

# 2. Robustly map the label column
label_col = None
for col in ['label', 'class', 'category', 'bonafide', 'target', 'spoof']:
    if col in df.columns:
        label_col = col
        break
if label_col is None:
    raise ValueError("Could not automatically detect label column! Please check column names.")
print(f"Using label column: '{label_col}'")
df = df.rename(columns={label_col: 'label'})

# 3. Clean nulls & empty texts
df = df.dropna(subset=['text', 'label'])
df = df[df['text'].str.strip().astype(bool)]

# 4. Normalize labels to 0 (Real/Bonafide) and 1 (Spoof/Fake)
def map_label(val):
    if isinstance(val, str):
        v = val.strip().lower()
        if 'bonafide' in v or v == 'real' or v == '0':
            return 0
        elif 'spoof' in v or v == 'fake' or v == '1' or 'ai' in v:
            return 1
    elif isinstance(val, (int, float)):
        if int(val) == 0: return 0
        elif int(val) == 1: return 1
    return 1 # Default fallback to spoof if ambiguous

df['label'] = df['label'].apply(map_label)

print("\n⚖️ Class Distribution:")
counts = df['label'].value_counts()
for label, count in counts.items():
    name = "Bonafide" if label == 0 else "Spoof"
    print(f"  - Class {label} [{name}]: {count:,} ({count/len(df)*100:.2f}%)")

# 5. Create Train (80%), Dev (10%), Eval (10%) splits stratified
if df['label'].nunique() > 1:
    train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])
    dev_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])
else:
    train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
    dev_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"\n📊 Data Splits -> Train: {len(train_df):,} | Dev: {len(dev_df):,} | Eval: {len(test_df):,}")

## Tokenize & Prepare Datasets

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)
test_dataset = Dataset.from_pandas(test_df)

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
dev_dataset = dev_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format columns for PyTorch
cols_to_remove = ['filename', 'text', 'split', '__index_level_0__']
train_dataset = train_dataset.remove_columns([c for c in cols_to_remove if c in train_dataset.column_names])
train_dataset = train_dataset.rename_column("label", "labels")
train_dataset.set_format("torch")

dev_dataset = dev_dataset.remove_columns([c for c in cols_to_remove if c in dev_dataset.column_names])
dev_dataset = dev_dataset.rename_column("label", "labels")
dev_dataset.set_format("torch")

test_dataset = test_dataset.remove_columns([c for c in cols_to_remove if c in test_dataset.column_names])
test_dataset = test_dataset.rename_column("label", "labels")
test_dataset.set_format("torch")

print("✅ Datasets tokenized and formatted for PyTorch training.")

## Compute Metrics (EER & AUC)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    auc = roc_auc_score(labels, probs)
    
    fpr, tpr, _ = roc_curve(labels, probs)
    fnr = 1 - tpr
    try:
        eer = brentq(lambda x: interp1d(fpr, fnr - fpr)(x), 0, 1)
    except:
        eer = np.mean(np.abs(fnr - fpr))
        
    return {"accuracy": acc, "f1": f1, "auc": auc, "eer": eer}

## Fine-Tune DistilBERT

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,      # Double batch size for faster epochs
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True if device == "cuda" else False, # Mixed precision speed boost on GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

print(f"🚀 Starting {MODEL_NAME} fine-tuning on device: {device}...")
trainer.train()

## Save Model & Evaluate

In [ ]:
print(f"Saving fine-tuned model to {MODEL_SAVE_DIR}...")
trainer.save_model(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))

eval_results = trainer.evaluate(eval_dataset=test_dataset)
print("\n🏆 Test Set Results:")
print(f"  - Accuracy: {eval_results['eval_accuracy']*100:.2f}%")
print(f"  - EER:      {eval_results['eval_eer']*100:.2f}%")
print(f"  - AUC:      {eval_results['eval_auc']:.4f}")
print(f"  - F1 Score: {eval_results['eval_f1']:.4f}")